# N-gram Analysis: Title and Abstract Corpus

This notebook contains the complete data-cleaning and embedding code. Select **Python (batill)**. Sections 1–6 clean the Scopus export; sections 7–9 embed the **full corpus of 3,819 papers** with `BAAI/bge-m3`. N-gram analysis will follow later.

The original CSV is preserved. Rerunning cleaning updates `data/abstracts/`; each embedding run creates its own output folder. You can execute sections 7–9 directly using the existing cleaned CSV.

## 1. Imports and file paths

Load the libraries and select the input and output paths. The input file contains 4,583 records. For another file with the same Scopus schema, adjust `SOURCE` and `OUTPUT_DIR`.

In [1]:
from __future__ import annotations

from collections import Counter
import csv
from datetime import date
import hashlib
import html
import json
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

In [2]:
# Find the project folder even if Jupyter was started in a subfolder.
ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data/bibliography/All_private_equity_bib.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start the notebook in the batill project folder.")

SOURCE = ROOT / "data/bibliography/All_private_equity_bib.csv"
OUTPUT_DIR = ROOT / "data/abstracts"
print(f"Input: {SOURCE}")
print(f"Output: {OUTPUT_DIR}")

Input: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/data/bibliography/All_private_equity_bib.csv
Output: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/data/abstracts


## 2. Cleaning rules and text preparation

Keep the abstract and bibliographic metadata needed to identify each record. Exclude missing abstracts, publisher-notice-only entries, records marked **Retracted**, and publication years later than the current calendar year. The year cutoff is recorded in the cleaning summary (2026 for this run).

Among eligible records, keep the first source occurrence per nonempty normalized DOI and exclude later occurrences, even when their abstracts differ. Also exclude repeated exact cleaned text. Records without a DOI can stay. Short abstracts are retained without a length warning. Errata are retained and flagged because they are corrections, not retractions. Shared titles alone do not cause exclusion.

Decode HTML entities, remove known formatting tags and invisible text artifacts, and normalize whitespace. Remove appended copyright/licensing notices and preserve them in the audit. Keep capitalization, punctuation, numbers, stopwords, word order, and statistical comparisons such as `P <.001`. No language or subject filtering is applied.

For embedding, combine the normalized title and cleaned abstract into `embedding_text`, separated by a blank line. This gives the model both parts in one input per paper. If a title is missing, use the abstract alone. Keep `title` and `abstract` separately as well.

In [3]:
FIELDS = {
    "paper_id": "EID",
    "title": "Title",
    "authors": "Authors",
    "year": "Year",
    "journal": "Source title",
    "doi": "DOI",
    "url": "Link",
    "document_language": "Language of Original Document",
    "document_type": "Document Type",
}
MISSING = {"", "[no abstract available]", "no abstract available", "n/a", "null"}
# Restrict markup recognition to actual formatting tags: <.001 is a p-value.
TAGS = re.compile(r"</?(?:p|br|div|span|b|i|em|strong|sub|sup|jats:p)\b[^>]*>", re.I)
NOTICE = re.compile(r"(?:\bCopyrights?\s*:?\s*(?:©|\d{4}|The Author)|©)", re.I)

In [4]:
def normalize(value: str) -> str:
    value = unicodedata.normalize("NFC", html.unescape(value))
    value = value.replace("\u00ad", "").replace("\u200b", "").replace("\ufeff", "")
    return " ".join(value.split())


def clean_abstract(value: str) -> tuple[str, str]:
    """Keep prose intact; return the cleaned text and removed trailing notice."""
    value = normalize(TAGS.sub(" ", value))
    if value.casefold() in MISSING:
        return "", ""
    if value.startswith("Apart from any fair dealing for the purposes of research or private study,") and "Copyright, Designs and Patents Act 1988" in value:
        return "", value
    match = NOTICE.search(value)
    if match:
        # Scopus appends copyright/licensing statements. Preserve every removed
        # suffix in the audit so the decision can be inspected and reversed.
        return value[:match.start()].rstrip(), value[match.start():]
    return value, ""


def write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

## 3. Build the corpus and record cleaning decisions

Define the function that validates the CSV schema and Scopus IDs, cleans the abstracts, and writes four files:

- `abstracts_clean.csv`: abstracts, essential metadata, and combined `embedding_text`.
- `cleaning_audit.csv`: the decision for every source record, duplicate mappings, and removed publisher notices.
- `review.csv`: retained records with review flags.
- `cleaning_summary.json`: rules, counts, publication-year cutoff, and the source file's SHA-256 checksum.

Check exclusions in this order: missing/notice-only abstract, retracted document type, future publication year, repeated DOI, repeated cleaned text. Each excluded record receives one reason. Only retained records reserve a DOI or abstract, so a rejected record cannot displace an eligible one. DOI matching ignores capitalization and common DOI URL prefixes; empty DOIs do not match each other.

Different abstracts sharing a title, invalid publication years, and errata remain flagged for review. Short abstracts are accepted without a length flag. The publication language may differ from the abstract language. Retraction filtering uses the CSV's document-type field; no external retraction lookup is performed.

In [5]:
def prepare(source: Path, destination: Path) -> dict:
    csv.field_size_limit(100_000_000)  # Large reference fields in the raw export.
    with source.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"Abstract", *FIELDS.values()}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Missing Scopus columns: {sorted(missing)}")
        source_columns = list(reader.fieldnames)
        records = list(reader)
    if any(None in row or any(v is None for v in row.values()) for row in records):
        raise ValueError("Malformed CSV: at least one record has the wrong field count")
    ids = [normalize(row["EID"]) for row in records]
    if any(not key for key in ids) or len(ids) != len(set(ids)):
        raise ValueError("Scopus EIDs must be nonempty and unique for traceability")

    cutoff_year = date.today().year
    corpus, audit, seen_text, seen_doi = [], [], {}, {}
    for number, raw in enumerate(records, 1):
        abstract, notice = clean_abstract(raw["Abstract"])
        row = {field: normalize(raw[column]) for field, column in FIELDS.items()}
        row["doi"] = re.sub(r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)", "", row["doi"], flags=re.I).lower()
        row.update(source_record=number, abstract=abstract, abstract_word_count=len(abstract.split()))
        row["embedding_text"] = "\n\n".join(part for part in (row["title"], abstract) if part)
        # Apply eligibility rules before deduplication. Only retained records
        # reserve a DOI or text, so excluded records cannot suppress valid ones.
        status, reason, duplicate_of = "kept", "", ""
        if not abstract:
            status, reason = "excluded", "publisher_notice_only" if notice else "missing_abstract"
        elif row["document_type"].casefold() == "retracted":
            status, reason = "excluded", "retracted"
        elif row["year"].isdigit() and int(row["year"]) > cutoff_year:
            status, reason = "excluded", "future_publication_year"
        elif row["doi"] and row["doi"] in seen_doi:
            status, reason = "excluded", "duplicate_doi"
            duplicate_of = seen_doi[row["doi"]]
        elif abstract in seen_text:
            status, reason = "excluded", "duplicate_cleaned_abstract"
            duplicate_of = seen_text[abstract]
        else:
            seen_text[abstract] = row["paper_id"]
            if row["doi"]:
                seen_doi[row["doi"]] = row["paper_id"]
            corpus.append(row)
        audit.append({
            "source_record": number, "paper_id": row["paper_id"],
            "title": row["title"], "status": status, "reason": reason,
            "duplicate_of": duplicate_of,
            "abstract_changed": abstract != raw["Abstract"],
            "removed_notice": notice,
        })

    title_counts = Counter(row["title"].casefold() for row in corpus if row["title"])
    review = []
    for row in corpus:
        flags = []
        if row["title"] and title_counts[row["title"].casefold()] > 1:
            flags.append("shared_title_different_abstract")
        if not row["year"].isdigit():
            flags.append("invalid_year")
        if row["document_type"].casefold() == "erratum":
            flags.append("erratum")
        if "©" in row["abstract"]:
            flags.append("remaining_copyright_symbol")
        if flags:
            review.append({**row, "review_flags": ";".join(flags)})

    fields = ["paper_id", "source_record", *[k for k in FIELDS if k != "paper_id"], "abstract", "abstract_word_count", "embedding_text"]
    outputs = {"abstracts_clean.csv", "cleaning_audit.csv", "review.csv", "cleaning_summary.json"}
    if source.resolve() in {(destination / name).resolve() for name in outputs}:
        raise ValueError("Output would overwrite the source CSV")
    destination.mkdir(parents=True, exist_ok=True)
    write_csv(destination / "abstracts_clean.csv", corpus, fields)
    write_csv(destination / "cleaning_audit.csv", audit, ["source_record", "paper_id", "title", "status", "reason", "duplicate_of", "abstract_changed", "removed_notice"])
    write_csv(destination / "review.csv", review, [*fields, "review_flags"])
    summary = {
        "source": str(source.resolve()),
        "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
        "publication_year_cutoff": cutoff_year,
        "embedding_text_column": "embedding_text",
        "source_rows": len(records), "source_column_count": len(source_columns),
        "retained_rows": len(corpus), "retained_columns": fields,
        "exclusions": dict(Counter(row["reason"] for row in audit if row["status"] == "excluded")),
        "removed_publisher_notices": sum(bool(row["removed_notice"]) for row in audit),
        "review_rows": len(review),
        "review_flags": dict(Counter(flag for row in review for flag in row["review_flags"].split(";"))),
        "retained_document_languages": dict(Counter(row["document_language"] for row in corpus)),
        "retained_document_types": dict(Counter(row["document_type"] for row in corpus)),
        "rules": [
            "Source preserved; source_record is the 1-based CSV data record, not a physical line number.",
            "Decode HTML entities; remove known formatting tags and invisible artifacts; normalize whitespace and Unicode NFC.",
            "Remove trailing copyright/licensing notices, preserving each removed suffix in the audit.",
            "Exclude missing/placeholder abstracts, notice-only entries and repeated exact cleaned abstracts; retain first source occurrence.",
            "Exclude records marked Retracted and publication years later than the current year, recorded as publication_year_cutoff.",
            "Among eligible records retain the first source occurrence per nonempty normalized DOI; audit later occurrences as duplicate_doi.",
            "Exclusion precedence: missing/notice-only abstract, retracted, future year, duplicate DOI, duplicate text. Only retained records reserve identifiers/text.",
            "Keep short abstracts without a length warning; keep missing DOIs. Flag shared titles, invalid years and errata for review.",
            "Keep all languages and subjects; remaining review flags are not exclusions.",
            "Document language describes the publication and need not be the abstract language.",
            "Preserve case, punctuation, numbers, stopwords and word order; no stemming, lemmatization or n-gram tokenization.",
            "Use embedding_text for vector embedding: normalized title, blank line, cleaned abstract. If title is empty, use the abstract alone.",
            "Keep title and abstract separately for inspection and later n-gram analysis; other metadata is not included in embedding_text.",
        ],
    }
    (destination / "cleaning_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return summary

## 4. Run the cleaning

Apply the cleaning function to the source CSV, save the four output files, and display the record counts and exclusion reasons.

In [6]:
summary = prepare(SOURCE, OUTPUT_DIR)
display(pd.DataFrame([
    {"Metric": "Source records", "Count": summary["source_rows"]},
    {"Metric": "Cleaned abstracts", "Count": summary["retained_rows"]},
    *[{"Metric": f"Excluded: {reason}", "Count": count}
      for reason, count in summary["exclusions"].items()],
    {"Metric": "Removed publisher notices", "Count": summary["removed_publisher_notices"]},
    {"Metric": "Retained records for review", "Count": summary["review_rows"]},
]))

,Metric,Count
0,Source records,4583
1,Cleaned abstracts,3819
2,Excluded: missing_abstract,695
3,Excluded: future_publication_year,1
4,Excluded: duplicate_cleaned_abstract,18
5,Excluded: duplicate_doi,44
6,Excluded: retracted,5
7,Excluded: publisher_notice_only,1
8,Removed publisher notices,3036
9,Retained records for review,86


## 5. Inspect results and prepare embedding inputs

Load the generated tables and preview the cleaned records and review flags. Use `embedding_text` for vector embedding: it contains the title followed by a blank line and the abstract. Other metadata is excluded from this text. The title and abstract also remain in their own columns for inspection and later n-gram analysis.

`embedding_texts` contains one combined input per paper. `embedding_paper_ids` preserves the matching IDs in the same order for assigning the vectors back to papers. This step prepares the inputs; it does not run an embedding model.

In [7]:
abstracts = pd.read_csv(OUTPUT_DIR / "abstracts_clean.csv", keep_default_na=False)
review = pd.read_csv(OUTPUT_DIR / "review.csv", keep_default_na=False)
audit = pd.read_csv(OUTPUT_DIR / "cleaning_audit.csv", keep_default_na=False)
display(abstracts.head())
display(review[["paper_id", "title", "review_flags"]].head(10))

embedding_paper_ids = abstracts["paper_id"].tolist()
embedding_texts = abstracts["embedding_text"].tolist()
print(f"Prepared {len(embedding_texts):,} title-and-abstract inputs for embedding.")
print(embedding_texts[0] if embedding_texts else "No embedding inputs available.")

Prepared 3,819 title-and-abstract inputs for embedding.
Risk-Adjusted Returns of Private Equity Funds: A New Approach

This paper introduces a new metric, α, to benchmark the performance of individual private equity funds. Our metric is substantially less sensitive to noise in fund cash flows compared to the popular public market equivalent (PME) and its generalization (GPME), while having the same aggregate pricing implications as GPME. For a large data set of fund cash flows, α estimates have much lower standard deviation across funds than does (G)PME. For buyout funds, PME and α are close, but deviate in certain subsamples. Using α increases power in regressions involving fund performance and improves performance predictability of future funds.


,paper_id,source_record,title,authors,year,journal,doi,url,document_language,document_type,abstract,abstract_word_count,embedding_text
0,2-s2.0-105013645261,1,Risk-Adjusted Returns of Private Equity Funds:...,Korteweg A.; Nagel S.,2025,Review of Financial Studies,10.1093/rfs/hhae067,https://www.scopus.com/pages/publications/1050...,English,Article,"This paper introduces a new metric, α, to benc...",99,Risk-Adjusted Returns of Private Equity Funds:...
1,2-s2.0-105009828648,3,Decoding the Material Influence Test: Implicat...,Chauhan S.; Mehta D.,2025,Antitrust Bulletin,10.1177/0003603x251346148,https://www.scopus.com/pages/publications/1050...,English,Review,The Competition Commission of India (CCI) has ...,145,Decoding the Material Influence Test: Implicat...
2,2-s2.0-85211604820,4,Do private equity investors value ESG and SDGs...,Ben Noamene T.,2025,Society and Business Review,10.1108/sbr-09-2024-0303,https://www.scopus.com/pages/publications/8521...,English,Article,Purpose This study aims to explore the percept...,232,Do private equity investors value ESG and SDGs...
3,2-s2.0-105040531495,5,Credit ratings and capital providers,Syed Z.A.; Eskandari R.; Yazdifar H.,2026,International Journal of Managerial Finance,10.1108/ijmf-12-2024-0695,https://www.scopus.com/pages/publications/1050...,English,Article,Purpose – We study how credit ratings shape fi...,228,Credit ratings and capital providers\n\nPurpos...
4,2-s2.0-105022824881,6,Piercing the holding veil to enter family capi...,Belot Couloumies Q.; Baud C.,2025,Socio-Economic Review,10.1093/ser/mwaf005,https://www.scopus.com/pages/publications/1050...,English,Article,Through a longitudinal case study spanning 196...,140,Piercing the holding veil to enter family capi...


,paper_id,title,review_flags
0,2-s2.0-105030415497,Correction: Private equity and reproductive me...,erratum
1,2-s2.0-105050420082,Correction: Hospital Staffing and Patient Outc...,erratum
2,2-s2.0-105033570572,Private equity,shared_title_different_abstract
3,2-s2.0-85209884767,Correction: Assessing risk of health outcomes ...,erratum
4,2-s2.0-105017965420,Start-Up Space: Global Investment Trends,shared_title_different_abstract
5,2-s2.0-85210172435,Corrigendum: The predictive validity of a Brai...,erratum
6,2-s2.0-85189672482,Corrigendum to “Plasma growth differentiation ...,erratum
7,2-s2.0-85197972908,Erratum: Geographic penetration of private equ...,erratum
8,2-s2.0-105000015788,Correction to: Private Equity’s Growing Presen...,erratum
9,2-s2.0-85204246719,Start-Up Space: Global Investment Trends,shared_title_different_abstract


## 6. Check data integrity

Verify that every source record is represented in the audit, retained and excluded counts add up, texts and IDs are unique, and duplicate mappings point to retained records. Check that every nonempty DOI is unique, no future publication years or records marked Retracted remain, and short abstracts are not flagged for length. Also check that flagged records belong to the retained corpus and that the source file's checksum has not changed during cleaning.

Verify that each embedding input matches its title and abstract, that no combined text is empty, and that the embedding IDs and texts stay aligned.

In [8]:
assert len(abstracts) == summary["retained_rows"]
assert len(audit) == summary["source_rows"]
assert len(abstracts) + sum(summary["exclusions"].values()) == len(audit)
assert abstracts["paper_id"].is_unique
assert abstracts["abstract"].is_unique
assert abstracts["abstract"].str.strip().ne("").all()
assert not abstracts["abstract"].isin(["[No abstract available]"]).any()
assert set(audit.loc[audit["status"] == "kept", "paper_id"]) == set(abstracts["paper_id"])
assert set(audit.loc[audit["duplicate_of"] != "", "duplicate_of"]) <= set(abstracts["paper_id"])
assert set(review["paper_id"]) <= set(abstracts["paper_id"])
assert len(review) == summary["review_rows"]
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == summary["source_sha256"]
assert abstracts.loc[abstracts["doi"] != "", "doi"].is_unique
publication_years = pd.to_numeric(abstracts["year"], errors="coerce")
assert not (publication_years > summary["publication_year_cutoff"]).any()
assert not abstracts["document_type"].str.casefold().eq("retracted").any()
assert not review["review_flags"].str.contains("short_abstract", regex=False).any()
expected_embedding_texts = [
    "\n\n".join(part for part in (title, abstract) if part)
    for title, abstract in zip(abstracts["title"], abstracts["abstract"])
]
assert embedding_texts == expected_embedding_texts
assert all(text.strip() for text in embedding_texts)
assert embedding_paper_ids == abstracts["paper_id"].tolist()
assert len(embedding_paper_ids) == len(embedding_texts) == summary["retained_rows"]
print(f"All checks passed: {len(abstracts):,} abstracts saved.")

All checks passed: 3,819 abstracts saved.


## 7. Configure BGE-M3 for the full corpus

Use [`BAAI/bge-m3`](https://huggingface.co/BAAI/bge-m3) through Sentence Transformers. It supports **8,192 input tokens** and produces **1,024-dimensional dense vectors**. Its supplied pooling configuration uses the first token; the vectors are L2-normalized for cosine similarity. No SPECTER2 adapters are used.

Each input is the existing `embedding_text`: title, blank line, and complete cleaned abstract. The code audits lengths using BGE-M3's own tokenizer, including special tokens. It stops if an input selected for inference exceeds the model limit. No truncation, splitting, or dropping of selected papers is allowed.

The default run embeds **all 3,819 cleaned papers** with `RUN_MODE = "full"`. `BATCH_SIZE = 8` limits memory usage while processing every paper. The earlier 50-paper benchmark used seed 42; it remains available as an optional sample mode. `RUN_EMBEDDING = True` starts the configured run when section 9 executes.

**Environment:** use the existing **Python (batill)** kernel in `.venv`, which already has Sentence Transformers, Transformers, and PyTorch. The SPECTER2 environment is no longer needed for this notebook. Model files are cached in `.cache/huggingface`; the first run downloads the model. The exact revision, package versions, input checksum, and device are recorded with every result.

In [1]:
RUN_EMBEDDING = True
RUN_MODE = "full"  # Embed every cleaned paper; the 50-paper benchmark is complete.
SAMPLE_SIZE = 50  # Used only if RUN_MODE is changed back to "sample".
SAMPLE_SEED = 42  # Used only in sample mode.
BATCH_SIZE = 8
DEVICE = "auto"  # "auto", "cpu", "mps" (Apple GPU), or "cuda".
MAX_TOKENS = 8192
MODEL_NAME = "BAAI/bge-m3"
MODEL_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"  # Exact revision tested in the 50-paper benchmark.

from pathlib import Path
import os

EMBED_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data/abstracts/abstracts_clean.csv").is_file()),
    None,
)
if EMBED_ROOT is None:
    raise FileNotFoundError("Run the cleaning first and open this notebook in the batill project.")
EMBED_SOURCE = EMBED_ROOT / "data/abstracts/abstracts_clean.csv"
EMBED_OUTPUT = EMBED_ROOT / "outputs/abstract_embeddings/bge_m3"
MODEL_CACHE = EMBED_ROOT / ".cache/huggingface/hub"
os.environ.setdefault("HF_HOME", str(EMBED_ROOT / ".cache/huggingface"))

'/Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/.cache/huggingface'

## 8. Define the embedding runner and timing

The runner saves the full-corpus length audit, exact sampled inputs, vectors, matching paper IDs, batch timings, and a run summary in `outputs/abstract_embeddings/bge_m3/<run>/`. Vector rows follow the order in `paper_ids.csv` and `selected_papers.csv`.

Timing separates these stages:

1. Tokenizer download and length audit.
2. Model download and loading.
3. One warm-up batch, timed separately.
4. Embedding every selected paper, including batch tokenization, inference, normalization, and transfer of vectors to CPU.

Accelerator synchronization ensures that the measured inference time includes completed GPU work. The warm-up papers are re-embedded in the measured run, so the reported inference time covers the complete selected corpus. A full-corpus estimate is a rough extrapolation, not a measured full run: document length, batch padding, and hardware load affect it.

Progress and batch timings are saved as the run proceeds. Finite values, vector dimensions, unit norms, and saved row alignment are checked. If memory runs out, reduce `BATCH_SIZE`. The code does not silently change devices or reduce text length.

In [2]:
def run_bge_m3_embeddings():
    import hashlib
    import json
    import platform
    from datetime import datetime, timezone
    from importlib.metadata import version
    from io import BytesIO
    from time import perf_counter
    from uuid import uuid4

    import numpy as np
    import pandas as pd
    import torch
    from huggingface_hub import HfApi
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer

    if RUN_MODE not in {"sample", "full"}:
        raise ValueError("RUN_MODE must be 'sample' or 'full'.")
    if not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be a positive integer.")
    if RUN_MODE == "sample" and (not isinstance(SAMPLE_SIZE, int) or SAMPLE_SIZE < 1):
        raise ValueError("SAMPLE_SIZE must be a positive integer.")
    if MODEL_NAME != "BAAI/bge-m3" or MAX_TOKENS != 8192:
        raise ValueError("This runner expects BAAI/bge-m3 with its 8192-token limit.")

    device = DEVICE
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else (
            "mps" if torch.backends.mps.is_available() else "cpu"
        )
    if device not in {"cpu", "mps", "cuda"}:
        raise ValueError("DEVICE must be 'auto', 'cpu', 'mps', or 'cuda'.")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Select an available device.")
    if device == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS is unavailable. Select an available device.")

    def synchronize():
        if device == "cuda":
            torch.cuda.synchronize()
        elif device == "mps":
            torch.mps.synchronize()

    started = perf_counter()
    source_bytes = EMBED_SOURCE.read_bytes()
    papers = pd.read_csv(BytesIO(source_bytes), keep_default_na=False, dtype=str)
    required = {"paper_id", "title", "abstract", "embedding_text"}
    if not required.issubset(papers.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(papers.columns))}")
    if papers.empty or not papers["paper_id"].is_unique:
        raise ValueError("The corpus must be nonempty with unique paper IDs.")
    for name in ["paper_id", "abstract", "embedding_text"]:
        if papers[name].str.strip().eq("").any():
            raise ValueError(f"Empty {name} values found; rerun cleaning.")
    expected_texts = ["\n\n".join(part for part in (title, abstract) if part)
                      for title, abstract in zip(papers["title"], papers["abstract"])]
    if papers["embedding_text"].tolist() != expected_texts:
        raise ValueError("embedding_text does not match title + blank line + abstract.")
    if RUN_MODE == "sample" and SAMPLE_SIZE > len(papers):
        raise ValueError("SAMPLE_SIZE exceeds the number of available papers.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run_dir = EMBED_OUTPUT / f"{stamp}_{RUN_MODE}_{uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    summary = {
        "status": "started", "source": str(EMBED_SOURCE),
        "source_sha256": hashlib.sha256(source_bytes).hexdigest(),
        "run_mode": RUN_MODE, "sample_seed": SAMPLE_SEED,
        "requested_sample_size": SAMPLE_SIZE if RUN_MODE == "sample" else None,
        "corpus_size": len(papers), "device": device, "batch_size": BATCH_SIZE,
        "max_tokens": MAX_TOKENS, "truncation": False, "chunking": False,
        "model": MODEL_NAME, "input_column": "embedding_text",
        "input_format": "title + blank line + abstract", "pooling": "model-defined CLS",
        "normalization": "L2", "dtype": "float32",
        "python": platform.python_version(), "platform": platform.platform(),
        "packages": {name: version(name) for name in
                     ["torch", "sentence-transformers", "transformers", "huggingface-hub", "numpy", "pandas"]},
    }

    def save_summary():
        (run_dir / "run_summary.json").write_text(
            json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
        )

    save_summary()
    print(f"Run folder: {run_dir}", flush=True)
    print(f"Device: {device}; mode: {RUN_MODE}", flush=True)
    try:
        audit_start = perf_counter()
        commit = HfApi().model_info(MODEL_NAME, revision=MODEL_REVISION).sha
        summary["model_revision"] = commit
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=commit, cache_dir=MODEL_CACHE)
        counts = []
        for start in range(0, len(papers), 128):
            encoded = tokenizer(
                papers["embedding_text"].iloc[start:start + 128].tolist(),
                truncation=False, padding=False, add_special_tokens=True,
                return_token_type_ids=False, verbose=False,
            )
            counts.extend(len(ids) for ids in encoded["input_ids"])
        papers["token_count"] = counts
        papers["over_limit"] = papers["token_count"] > MAX_TOKENS
        papers[["paper_id", "title", "token_count", "over_limit"]].to_csv(
            run_dir / "token_lengths.csv", index=False
        )
        selected = (papers.sample(n=SAMPLE_SIZE, random_state=SAMPLE_SEED, replace=False)
                    if RUN_MODE == "sample" else papers).copy().reset_index(drop=True)
        selected.insert(0, "embedding_row", np.arange(len(selected)))
        selected.to_csv(run_dir / "selected_papers.csv", index=False)
        summary.update(
            selected_size=len(selected), corpus_over_limit=int(papers["over_limit"].sum()),
            selected_over_limit=int(selected["over_limit"].sum()),
            maximum_corpus_tokens=int(papers["token_count"].max()),
            maximum_sample_tokens=int(selected["token_count"].max()),
            tokenizer_and_audit_seconds=perf_counter() - audit_start,
        )
        save_summary()
        print(f"Over {MAX_TOKENS} tokens: {summary['corpus_over_limit']}/{len(papers)} corpus inputs; "
              f"{summary['selected_over_limit']}/{len(selected)} selected inputs.", flush=True)
        if summary["selected_over_limit"]:
            raise ValueError("Selected inputs exceed the context window. No text was truncated and no vectors were computed.")

        summary["status"] = "loading_model"
        save_summary()
        print("Loading BGE-M3 (the first run downloads model weights)...", flush=True)
        loading_start = perf_counter()
        model = SentenceTransformer(
            MODEL_NAME, revision=commit, device=device, cache_folder=str(MODEL_CACHE),
            model_kwargs={"dtype": torch.float32}, trust_remote_code=False,
        )
        if model.max_seq_length < MAX_TOKENS:
            raise ValueError(f"Loaded model limit {model.max_seq_length} is below {MAX_TOKENS}.")
        if model.get_sentence_embedding_dimension() != 1024:
            raise ValueError("Unexpected BGE-M3 embedding dimension.")
        model.eval()
        synchronize()
        summary["model_load_seconds"] = perf_counter() - loading_start
        texts = selected["embedding_text"].tolist()

        def encode_batch(batch):
            return model.encode(
                batch, batch_size=BATCH_SIZE, show_progress_bar=False, prompt="",
                convert_to_numpy=True, normalize_embeddings=True, precision="float32",
            )

        with torch.inference_mode():
            summary["status"] = "warming_up"
            save_summary()
            print("Warming up one batch...", flush=True)
            synchronize()
            warmup_start = perf_counter()
            encode_batch(texts[:min(BATCH_SIZE, len(texts))])
            synchronize()
            summary["warmup_seconds"] = perf_counter() - warmup_start
            summary["status"] = "embedding"
            save_summary()
            batches, timings = [], []
            inference_start = perf_counter()
            for start in range(0, len(texts), BATCH_SIZE):
                batch = texts[start:start + BATCH_SIZE]
                synchronize()
                batch_start = perf_counter()
                batches.append(encode_batch(batch))
                synchronize()
                elapsed = perf_counter() - batch_start
                timings.append({"start_row": start, "papers": len(batch), "seconds": elapsed})
                pd.DataFrame(timings).to_csv(run_dir / "batch_timings.csv", index=False)
                print(f"Embedded {start + len(batch)}/{len(texts)} papers ({elapsed:.2f} s/batch).", flush=True)
            inference_seconds = perf_counter() - inference_start

        vectors = np.concatenate(batches, axis=0).astype(np.float32, copy=False)
        if vectors.shape != (len(selected), 1024):
            raise ValueError(f"Unexpected embedding shape: {vectors.shape}")
        if not np.isfinite(vectors).all():
            raise ValueError("Non-finite embedding values detected.")
        if not np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5):
            raise ValueError("Embedding normalization check failed.")
        np.save(run_dir / "embeddings.npy", vectors, allow_pickle=False)
        selected[["embedding_row", "paper_id"]].to_csv(run_dir / "paper_ids.csv", index=False)
        saved_ids = pd.read_csv(run_dir / "paper_ids.csv", dtype={"paper_id": str})
        if saved_ids["paper_id"].tolist() != selected["paper_id"].tolist():
            raise ValueError("Saved paper IDs do not match the vector row order.")
        if not np.array_equal(np.load(run_dir / "embeddings.npy", allow_pickle=False), vectors):
            raise ValueError("Saved vectors differ from in-memory vectors.")
        summary.update(
            status="complete", shape=list(vectors.shape), inference_seconds=inference_seconds,
            papers_per_second=len(selected) / inference_seconds,
            approximate_full_corpus_inference_minutes=(inference_seconds / len(selected)) * len(papers) / 60,
            estimate_caveat="Rough extrapolation; excludes loading/warm-up and depends on lengths, padding and hardware.",
            total_seconds=perf_counter() - started,
        )
        save_summary()
        print(f"Saved {vectors.shape[0]} vectors of dimension {vectors.shape[1]}.", flush=True)
        print(f"Model download/loading: {summary['model_load_seconds']:.2f} s; "
              f"warm-up: {summary['warmup_seconds']:.2f} s; "
              f"embedding {len(selected)} papers: {inference_seconds:.2f} s.", flush=True)
        print(f"Total run: {summary['total_seconds']:.2f} s.", flush=True)
        if RUN_MODE == "sample":
            estimate = summary["approximate_full_corpus_inference_minutes"]
            print(f"Rough full-corpus inference estimate: {estimate:.1f} minutes, excluding setup.", flush=True)
        return {"run_dir": run_dir, "summary": summary}
    except Exception as exc:
        summary.update(status="failed", error_type=type(exc).__name__, error=str(exc),
                       total_seconds=perf_counter() - started)
        save_summary()
        print(f"Run stopped. Diagnostics saved to {run_dir}", flush=True)
        raise

## 9. Run the full corpus

Select **Python (batill)** and execute sections 7–9 to embed the complete cleaned CSV. `RUN_MODE = "full"` is set. The output reports measured inference time separately from setup and warm-up.

Every attempt writes to a new folder, preserving earlier results and the benchmark. A full-corpus token-length audit is saved before inference.

**Full-corpus result (2026-09-30):** successfully embedded **all 3,819 papers** with BGE-M3 on the Mac GPU (`mps`, float32, batch size 8). Inference took **1185.11 seconds (19 minutes 45 seconds)**; total time including cached model loading, warm-up, token audit and writing outputs was **1196.76 seconds (19 minutes 57 seconds)**. The saved matrix has shape **3,819 × 1,024**. All vectors are finite and normalized; every source paper and complete title-and-abstract input is aligned with its saved vector. No truncation or chunking was used.

Full results: `outputs/abstract_embeddings/bge_m3/20260930T174845_508054Z_full_3c498a9d/`. `embeddings.npy` contains the matrix, `paper_ids.csv` maps vector rows to papers, and `selected_papers.csv` contains the exact inputs. `run_summary.json`, `token_lengths.csv`, and `batch_timings.csv` record provenance and timing. The 50-paper benchmark below is retained for comparison.

**Measured result (2026-09-30):** all 50 random papers embedded successfully on the Mac GPU (`mps`), using float32 and batch size 8. Embedding took **13.13 seconds**, excluding **104.03 seconds** for model download/loading and **5.24 seconds** for warm-up. Total run time including the token audit and output writing was **129.53 seconds**. Saved vectors have shape **50 × 1,024** and passed integrity checks.

All 3,819 corpus inputs fit the 8,192-token limit; the maximum was 1,973 tokens (598 in the sample). No truncation or chunking was used. Results: `outputs/abstract_embeddings/bge_m3/20260930T174040_070486Z_sample_29fced1c`. The measured sample throughput gives a rough full-corpus inference estimate of **16.7 minutes**, excluding setup; this is not a measured full-corpus runtime.

In [3]:
bge_m3_result = None
if RUN_EMBEDDING:
    bge_m3_result = run_bge_m3_embeddings()
else:
    print("BGE-M3 is disabled. Set RUN_EMBEDDING=True to run the configured embedding job.")

/Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run folder: /Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/outputs/abstract_embeddings/bge_m3/20260930T174845_508054Z_full_3c498a9d


Device: mps; mode: full


Over 8192 tokens: 0/3819 corpus inputs; 0/3819 selected inputs.


Loading BGE-M3 (the first run downloads model weights)...


...

Embedded 3816/3819 papers (1.20 s/batch).


Embedded 3819/3819 papers (2.81 s/batch).


Saved 3819 vectors of dimension 1024.


Model download/loading: 4.94 s; warm-up: 3.73 s; embedding 3819 papers: 1185.11 s.


Total run: 1196.76 s.
